In [22]:
import pandas as pd
import numpy as np

from tqdm.notebook import tqdm
import requests

import pickle

In [23]:
reference_path = "../../../results/road/freeflow/reference_survey.parquet"
calibration_path = "../../../results/road/freeflow/calibration_cache_survey_only_majors.pickle"
output_path = "../../../results/road/freeflow/output.parquet"

# routing_endpoint = "http://sma.univ-eiffel.fr:18054//router/road"
routing_endpoint = "http://localhost:8054/router/road"

departure_time = 4 * 3600
maximum_batch_size = 400

In [24]:
# Load reference
df_reference = pd.read_parquet(reference_path)

In [25]:
# Prepare requests
df_reference["request_index"] = np.arange(len(df_reference))

# Convert to requests
request_list = []

for index, row in df_reference.iterrows():
    request_list.append({
        "request_index": int(row["request_index"]),
        "origin_x": row["origin_x"],
        "origin_y": row["origin_y"],
        "destination_x": row["destination_x"],
        "destination_y": row["destination_y"],
        "departure_time_s": departure_time
    })

In [26]:
# Prepare querying
def query_requests(request_list, settings):
    df_response = []
    batch_index = 0

    while batch_index * maximum_batch_size < len(request_list):
        batch = request_list[batch_index * maximum_batch_size : (batch_index + 1) * maximum_batch_size]

        response = requests.post("http://localhost:8054/router/road", json = {
            "batch": batch,
            "freeflow": settings
        })

        df_response.append(pd.DataFrame.from_records(response.json()))
        batch_index += 1

    return pd.concat(df_response)

In [27]:
# Obtain settings
with open(calibration_path, "rb") as f:
    history = pickle.load(f)

objective = np.inf
settings = None

for item in history:
    if item["objective"] < objective:
        settings = item["settings"]

In [28]:
df_response = query_requests(request_list, settings)

In [29]:
df_response = pd.merge(
    df_response, df_reference[["request_index", "trip_id"]], on = "request_index")

df_response = df_response.drop(
    columns = ["request_index"])

In [30]:
df_response.to_parquet(output_path)